# CS-781 Phase 5 - Legal Statute Identification (LSI): all experiments in one notebook

**Setup on Kaggle:** Accelerator *GPU T4 x2*, Internet on, secret `HF` (your Hugging Face token), and the same inputs as Phase 4:
`export_gqa`, `export_mla`, `export_retnet` (datasets) and the `Phase1-Arch3` notebook output (contains `export_encdec`).
Run once with `SMOKE = True` in the last cell (~15 min pipeline check, prints the dataset statistics), then set `SMOKE = False`
and *Save Version -> Save & Run All* (~6 h).

**Model (shared by every run, so the comparison is about the adaptation technique and the backbone).**
Each document is cut into 512-token chunks; the model reads the **first chunks plus the document's final chunk** (up to 4 x 512 tokens).
Every chunk goes through the Phase-1 backbone (decoders: causal states; enc-dec: bidirectional encoder with a padding mask), token states
are projected to 384 dims and get a chunk-position embedding. Two heads are added:
(a) **label-wise attention** - every statute has its own query vector and attends over all tokens of the document (the evidence for
Section 302 and for Section 34 sits in different sentences), and (b) a global document vector (masked mean + attention pooling).
Loss: BCE with a dampened positive weight per statute, clip(sqrt(neg/pos), 1, 15), so rare statutes are not ignored; output biases start at
the label prior. Macro-F1 is the metric, so after training three decision rules are compared with a repeated 2-fold check on dev
(fixed 0.5, one tuned global threshold, per-statute thresholds shrunk towards the global one) and the best one is used. Every document gets at
least one statute. 75-minute budget per run with a cosine schedule that follows the budget; the best dev checkpoint is kept.

| GPU 0 | GPU 1 |
|---|---|
| full fine-tuning - GQA | partial fine-tuning (top 6 layers) - RetNet |
| **LoRMA** (compulsory) - GQA | Houlsby adapters - enc-dec |
| BitFit - MLA | AdaLoRA - RetNet |
| **knowledge distillation** (teacher = mean probabilities of the six runs, student = 8-layer GQA) | ablation: LoRMA-GQA reading only the last 4 chunks |

then a dev-selected **ensemble**. Every run writes `ls_<method>_<arch>/submission.zip`, `results.json` (incl. per-statute F1) and probabilities;
all zips are collected in `/kaggle/working/submissions/`. All adaptation methods are written from scratch in `ls_peft.py`.

In [1]:
import os
HF_TOKEN = None
for _name in ("HF", "HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(_name)
        if HF_TOKEN:
            break
    except Exception as e:
        print("secret", _name, "not available:", type(e).__name__)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("No HF token found")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
%%writefile ls_metrics.py
"""Phase 5 (LSI) metrics and decision rules - numpy only.
Macro-F1 follows evaluate_phase5.py / sklearn f1_score(average="macro", zero_division=0) over ALL nlab labels:
a label with no gold and no predicted positives scores 0 (it still counts in the average)."""
import numpy as np

GRID = np.round(np.concatenate([np.arange(0.02, 0.10, 0.01), np.arange(0.10, 0.90, 0.025)]), 4)


def to_matrix(label_lists, nlab):
    Y = np.zeros((len(label_lists), nlab), dtype=bool)
    for i, ls in enumerate(label_lists):
        for c in ls:
            if 0 <= c < nlab:
                Y[i, c] = True
    return Y


def per_label_f1(Y, P):
    tp = (Y & P).sum(0).astype(float)
    fp = (~Y & P).sum(0).astype(float)
    fn = (Y & ~P).sum(0).astype(float)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)


def macro_f1(Y, P):
    return float(per_label_f1(Y, P).mean())


def micro_f1(Y, P):
    tp = (Y & P).sum(); fp = (~Y & P).sum(); fn = (Y & ~P).sum()
    return float(2 * tp / max(1, 2 * tp + fp + fn))


def subset_acc(Y, P):
    return float((Y == P).all(1).mean())


def macro_pr(Y, P):
    tp = (Y & P).sum(0).astype(float); fp = (~Y & P).sum(0); fn = (Y & ~P).sum(0)
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return float(p.mean()), float(r.mean())


def decide(prob, thr, min_one=True):
    """prob (N,L), thr scalar or (L,). Every document gets at least one statute (the label with the highest prob/thr)."""
    thr = np.broadcast_to(np.asarray(thr, dtype=float), (prob.shape[1],))
    P = prob >= thr
    if min_one:
        empty = ~P.any(1)
        if empty.any():
            j = np.argmax(prob[empty] / np.maximum(thr, 1e-6), 1)
            P[np.where(empty)[0], j] = True
    return P


def tune_global(Y, prob, min_one=True):
    best = (-1.0, 0.5)
    for t in GRID:
        f = macro_f1(Y, decide(prob, t, min_one))
        if f > best[0] + 1e-12:
            best = (f, float(t))
    return best[1], best[0]


def tune_per_label(Y, prob, t_global, min_pos=3, shrink=4.0):
    """Per-label threshold maximising that label's F1 on the given docs; shrunk towards the global threshold
    (weight n_pos/(n_pos+shrink)); labels with fewer than min_pos positives keep the global threshold."""
    L = Y.shape[1]
    F = np.zeros((len(GRID), L))
    for gi, t in enumerate(GRID):
        F[gi] = per_label_f1(Y, prob >= t)
    tbest = GRID[F.argmax(0)]
    npos = Y.sum(0).astype(float)
    w = npos / (npos + shrink)
    thr = w * tbest + (1 - w) * t_global
    thr[npos < min_pos] = t_global
    return thr


def fit_rule(Y, prob, rule):
    tg, _ = tune_global(Y, prob)
    if rule == "global":
        return np.full(Y.shape[1], tg)
    if rule == "fixed":
        return np.full(Y.shape[1], 0.5)
    return tune_per_label(Y, prob, tg)


def honest(Y, prob, rules=("fixed", "global", "perlabel"), seed=0, folds=2, reps=2):
    """Threshold rule fitted on one part of dev and scored on the other (repeated): which rule really generalises."""
    out = {r: [] for r in rules}
    n = len(Y)
    for rep in range(reps):
        idx = np.random.RandomState(seed + rep).permutation(n)
        parts = np.array_split(idx, folds)
        for k in range(folds):
            te = parts[k]; tr = np.concatenate([parts[j] for j in range(folds) if j != k])
            for r in rules:
                thr = fit_rule(Y[tr], prob[tr], r)
                out[r].append(macro_f1(Y[te], decide(prob[te], thr)))
    return {r: float(np.mean(v)) for r, v in out.items()}


def choose(Y, prob):
    """Pick the threshold rule with the best honest score, refit it on the whole dev set."""
    h = honest(Y, prob)
    rule = max(h, key=h.get)
    thr = fit_rule(Y, prob, rule)
    P = decide(prob, thr)
    return dict(rule=rule, thr=thr, honest=h, macro=macro_f1(Y, P), micro=micro_f1(Y, P), subset=subset_acc(Y, P),
                macro_p_r=macro_pr(Y, P), P=P)


def freq_buckets(train_count, Y, P, edges=(0, 50, 200, 10 ** 9)):
    """Macro-F1 of dev labels grouped by how often they occur in train (rare / medium / frequent).
    train_count: (L,) counts or the (N, L) train label matrix."""
    cnt = np.asarray(train_count)
    cnt = cnt.sum(0) if cnt.ndim == 2 else cnt[:Y.shape[1]]
    f = per_label_f1(Y, P)
    present = Y.sum(0) > 0
    res = {}
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (cnt >= lo) & (cnt < hi) & present
        res[f"train_count[{lo},{hi if hi < 10 ** 9 else 'inf'})"] = (int(m.sum()), float(f[m].mean()) if m.any() else None)
    return res


Writing ls_metrics.py


In [3]:
%%writefile ls_lib.py
"""CS-781 Phase 5 (LSI) library: multi-label statute identification on long legal documents with a Phase-1 backbone.
doc -> whitespace-normalised -> head tokens (+ the final chunk) -> chunks of CHUNK tokens -> Phase-1 backbone per chunk
(+ adaptation method) -> token states projected to H dims + chunk-position embedding
-> (a) label-wise attention: every statute has its own query and attends over all tokens of the document
   (b) a global document vector (masked mean + attention pooling) -> linear
-> logits = (a) + (b), one per statute. Loss: BCE with dampened positive weights. Thresholds tuned on dev."""
import os, sys, json, math, time, glob, random, hashlib, types, zipfile, importlib.util, contextlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from ls_metrics import to_matrix, macro_f1, tune_global

PAD_ID = 1


# ====================================================================== Phase-1 models
def load_module_from_file(path, name):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()[:10]


def find_p1_export(arch, roots=("/kaggle/input", "/kaggle/working")):
    for r in roots:
        for root, dirs, files in os.walk(r, followlinks=True):
            if "config.json" in files and "model.pt" in files:
                try:
                    if json.load(open(os.path.join(root, "config.json"))).get("arch") == arch:
                        return Path(root)
                except Exception:
                    pass
    return None


def load_p1(export_dir):
    export_dir = Path(export_dir)
    cj = json.load(open(export_dir / "config.json"))
    mod = load_module_from_file(export_dir / "legal_arch.py", f"legal_arch_{cj['arch']}_{file_hash(export_dir / 'legal_arch.py')}")
    bb = mod.build_model(mod.ArchConfig(**cj["cfg"]))
    bb.load_state_dict(torch.load(export_dir / "model.pt", map_location="cpu", weights_only=False))
    return mod, cj["cfg"], bb


def blocks_of(bb):
    return bb.blocks if bb.kind == "decoder" else bb.enc_blocks


def final_norm_of(bb):
    return bb.norm_f if bb.kind == "decoder" else bb.enc_norm


def strip_backbone(bb):
    if bb.kind == "encdec":
        del bb.dec_blocks
        del bb.dec_norm
    bb.lm_head = nn.Identity()
    return bb


def replace_modules(bb, match, factory):
    n = 0
    for blk in blocks_of(bb):
        for name, mod in list(blk.named_modules()):
            if name and match(name, mod):
                parent_name, _, leaf = name.rpartition(".")
                parent = blk.get_submodule(parent_name) if parent_name else blk
                setattr(parent, leaf, factory(mod))
                n += 1
    return n


def replace_linears(bb, targets, factory):
    targets = set(targets)
    return replace_modules(bb, lambda n, m: isinstance(m, nn.Linear) and n.split(".")[-1] in targets, factory)


def make_student_bb(mod, cfg, full_bb, keep):
    cfg = dict(cfg)
    if full_bb.kind == "decoder":
        cfg["n_layers"] = len(keep)
    else:
        cfg["n_enc_layers"] = len(keep)
        cfg["n_dec_layers"] = 1
    st = mod.build_model(mod.ArchConfig(**cfg))
    st.tok_emb.load_state_dict(full_bb.tok_emb.state_dict())
    final_norm_of(st).load_state_dict(final_norm_of(full_bb).state_dict())
    for i, j in enumerate(keep):
        blocks_of(st)[i].load_state_dict(blocks_of(full_bb)[j].state_dict())
    return st, cfg


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


# ---- key-padding mask for the enc-dec encoder (Phase-1 encoder never saw padding)
_ENC_MASK = [None]


def _masked_enc_attn_forward(self, x, cos, sin, cache=None):
    arch = sys.modules[type(self).__module__]
    B, T, _ = x.shape
    q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    k = self.wk(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    v = self.wv(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    q = arch.apply_rope(q, cos[:T], sin[:T])
    k = arch.apply_rope(k, cos[:T], sin[:T])
    m = _ENC_MASK[0]
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if m is None else m[:, None, None, :T])
    return self.wo(o.transpose(1, 2).reshape(B, T, -1))


def patch_encoder_padding_mask(bb):
    if bb.kind != "encdec":
        return 0
    for blk in bb.enc_blocks:
        blk.attn.forward = types.MethodType(_masked_enc_attn_forward, blk.attn)
    return len(bb.enc_blocks)


def max_positions(bb):
    rc = getattr(bb, "rope_cos", None)
    return int(rc.shape[0]) if rc is not None else int(getattr(bb.cfg, "max_seq_len", 1024))


def _call_block(blk, h, cos, sin, mask):
    if mask is not None:
        _ENC_MASK[0] = mask
    return blk(h, cos, sin)


def run_backbone(bb, x, valid, train_from=0, ckpt=True):
    enc = bb.kind == "encdec"
    blocks, norm = (bb.enc_blocks, bb.enc_norm) if enc else (bb.blocks, bb.norm_f)
    cos, sin = bb.rope_cos, bb.rope_sin
    grad = torch.is_grad_enabled()
    if enc:
        _ENC_MASK[0] = valid
    try:
        with torch.set_grad_enabled(grad and bb.tok_emb.weight.requires_grad and train_from == 0):
            h = bb.tok_emb(x)
        for i, blk in enumerate(blocks):
            if grad and i >= train_from:
                h = checkpoint(_call_block, blk, h, cos, sin, valid if enc else None, use_reentrant=False) if ckpt else blk(h, cos, sin)
            else:
                with torch.no_grad():
                    h = blk(h, cos, sin)
        return norm(h)
    finally:
        if enc:
            _ENC_MASK[0] = None


# ====================================================================== data
class Doc:
    __slots__ = ("id", "labels", "ids", "tail", "nwords", "ntok_est", "key")


LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "gold", "y", "target")
TEXT_KEYS = ("text", "document", "facts", "case_text", "doc", "content")


def parse_lsi(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list):
            raw = next(iter(raw.values()))
        else:
            raw = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in raw.items()]
    out = []
    for i, r in enumerate(raw):
        did = r.get("id", r.get("case_id", r.get("name", i)))
        text = ""
        for k in TEXT_KEYS:
            if k in r and r[k]:
                text = r[k]
                break
        if isinstance(text, list):
            text = " ".join(map(str, text))
        lab = None
        for k in LABEL_KEYS:
            if k in r and r[k] is not None:
                lab = r[k]
                break
        if lab is not None and not isinstance(lab, list):
            lab = [lab]
        if lab is not None:
            lab = sorted({int(c) for c in lab})
        out.append((str(did), str(text or ""), lab))
    return out, raw


def tokenize(records, tokenizer, keep_tok, chunk, select="headtail", normalize=True):
    """headtail/head: keep the first keep_tok tokens (facts and issues come first) and, if the document is longer,
    its last chunk separately (the operative end often names the provisions applied).
    tail: keep the last keep_tok tokens only (ablation: does the end of the document carry the statute signal?)."""
    head_chars, tail_chars = keep_tok * 7, chunk * 10
    docs, heads, tails = [], [], []
    for did, text, lab in records:
        t = " ".join(text.split()) if normalize else text
        d = Doc()
        d.id, d.labels, d.nwords = did, (lab or []), len(t.split())
        d.key = hashlib.md5(t[:3000].encode("utf-8", "ignore")).hexdigest()
        if select == "tail":
            heads.append(t[-head_chars:]); tails.append("")
        else:
            heads.append(t[:head_chars]); tails.append(t[-tail_chars:] if len(t) > head_chars else "")
        docs.append(d)
    for i in range(0, len(docs), 2000):
        hb = tokenizer.encode_batch(heads[i:i + 2000])
        tb = tokenizer.encode_batch(tails[i:i + 2000])
        for d, eh, et in zip(docs[i:i + 2000], hb, tb):
            hid = eh.ids or [PAD_ID]
            d.ntok_est = len(hid)
            if select == "tail":
                d.ids, d.tail = np.asarray(hid[-keep_tok:], dtype=np.int32), None
            else:
                d.ids = np.asarray(hid[:keep_tok], dtype=np.int32)
                d.tail = np.asarray(et.ids[-chunk:], dtype=np.int32) if (len(hid) > keep_tok and et.ids) else None
    return docs


def doc_chunks(d, k, chunk, select="headtail"):
    """Up to k chunks as (token array, position id); position = chunk index from the start (0..6), and 7 marks
    the final chunk of a document that was cut (headtail), so the model knows it is the end, not a continuation.
    headtail: first k-1 chunks + final chunk when the document is longer than k chunks, else the whole document.
    head: first k chunks.  tail: last k chunks of the kept tail tokens (positions 7-k+1..7)."""
    ids = d.ids
    n = max(1, math.ceil(len(ids) / chunk))
    pieces = [ids[j * chunk:(j + 1) * chunk] for j in range(n)]
    if select == "tail":
        L = len(ids)
        pieces = [ids[max(0, L - (j + 1) * chunk): L - j * chunk] for j in range(min(n, k))][::-1]
        return [(p, 8 - len(pieces) + j) for j, p in enumerate(pieces)]
    if select == "head":
        return [(p, j) for j, p in enumerate(pieces[:k])]
    has_tail = d.tail is not None and len(d.tail) > 0
    if n + int(has_tail) <= k:
        return [(p, j) for j, p in enumerate(pieces)] + ([(d.tail, 7)] if has_tail else [])
    return [(p, j) for j, p in enumerate(pieces[:k - 1])] + [((d.tail if has_tail else pieces[-1]), 7)]


def make_batch(docs, k, chunk, select="headtail"):
    per = [doc_chunks(d, k, chunk, select) for d in docs]
    B, K = len(docs), max(len(p) for p in per)
    cmask = np.zeros((B, K), dtype=bool)
    pos = np.zeros((B, K), dtype=np.int64)
    rows = []
    for b, p in enumerate(per):
        for j, (ids, ps) in enumerate(p):
            cmask[b, j] = True
            pos[b, j] = ps
            rows.append(ids if len(ids) else np.asarray([PAD_ID], dtype=np.int32))
    Lm = max(len(r) for r in rows)
    x = np.full((len(rows), Lm), PAD_ID, dtype=np.int64)
    valid = np.zeros((len(rows), Lm), dtype=bool)
    for i, r in enumerate(rows):
        x[i, :len(r)] = r
        valid[i, :len(r)] = True
    return torch.from_numpy(x), torch.from_numpy(valid), torch.from_numpy(cmask), torch.from_numpy(pos)


# ====================================================================== model
class LSIModel(nn.Module):
    def __init__(self, bb, hcfg, nlab, prior):
        super().__init__()
        d, H, dr = bb.cfg.d_model, hcfg["hid"], hcfg["dropout"]
        self.bb, self.train_from, self.ckpt = bb, int(hcfg.get("train_from", 0)), bool(hcfg.get("ckpt", True)) and bb.kind != "encdec"
        self.proj = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, H), nn.GELU())
        self.pos = nn.Embedding(8, H)
        nn.init.normal_(self.pos.weight, std=0.02)
        self.drop = nn.Dropout(dr)
        # (a) label-wise attention
        self.lq = nn.Parameter(torch.randn(nlab, H) * 0.02)
        self.lw = nn.Parameter(torch.randn(nlab, H) * 0.02)
        # (b) global document vector
        self.tok_att = nn.Linear(H, 1)
        self.glob = nn.Sequential(nn.LayerNorm(2 * H), nn.Dropout(dr), nn.Linear(2 * H, nlab))
        nn.init.zeros_(self.glob[-1].weight)
        p = torch.as_tensor(prior, dtype=torch.float32).clamp(1e-4, 0.5)
        with torch.no_grad():
            self.glob[-1].bias.copy_(torch.log(p / (1 - p)))       # start at the label prior

    def forward(self, x, valid, cmask, pos):
        B, K = cmask.shape
        h = run_backbone(self.bb, x, valid, self.train_from, self.ckpt)
        t = self.proj(h.float())                                           # (N, T, H)
        N, T, H = t.shape
        pflat = pos[cmask]                                                 # positions of the real chunks, in row order
        t = t + self.pos(pflat)[:, None, :]
        tok = t.new_zeros(B * K, T, H)
        vm = valid.new_zeros(B * K, T)
        tok[cmask.view(-1)] = t
        vm[cmask.view(-1)] = valid
        tok, vm = tok.view(B, K * T, H), vm.view(B, K * T)
        tok = self.drop(tok)
        # (a) label-wise attention over all tokens
        sc = torch.einsum("bth,lh->blt", tok, self.lq) / math.sqrt(H)
        sc = sc.masked_fill(~vm[:, None, :], -1e4)
        ctx = torch.einsum("blt,bth->blh", sc.softmax(-1), tok)
        lw = (ctx * self.lw).sum(-1)
        # (b) global vector
        m = vm.unsqueeze(-1).float()
        mean = (tok * m).sum(1) / m.sum(1).clamp(min=1)
        a = self.tok_att(tok).squeeze(-1).masked_fill(~vm, -1e4).softmax(1)
        att = (a.unsqueeze(-1) * tok).sum(1)
        return (lw + self.glob(torch.cat([mean, att], -1))).float()


# ====================================================================== AMP
def get_amp(device):
    if device == "cuda":
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        return (lambda: torch.autocast(device_type="cuda", dtype=dt)), dt
    return (lambda: contextlib.nullcontext()), None


def make_scaler(device, amp_dtype):
    en = device == "cuda" and amp_dtype == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=en)


# ====================================================================== inference
@torch.no_grad()
def predict(model, docs, k, chunk, bs, device, amp, select="headtail"):
    model.eval()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i].ids))
    out = None
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        x, v, cm, pos = make_batch([docs[j] for j in idx], k, chunk, select)
        with amp():
            lg = model(x.to(device), v.to(device), cm.to(device), pos.to(device))
        p = torch.sigmoid(lg.float()).cpu().numpy()
        if out is None:
            out = np.zeros((len(docs), p.shape[1]), dtype=np.float32)
        out[idx] = p
    return out


def evaluate(model, docs, cfg, device, amp, nlab):
    p = predict(model, docs, cfg["k_infer"], cfg["chunk"], cfg["eval_bs"], device, amp, cfg["select"])
    Y = to_matrix([d.labels for d in docs], nlab)
    t, f = tune_global(Y, p)
    return dict(f1_global=f, thr=t), p


# ====================================================================== training
def train(model, train_docs, dev_docs, cfg, device, amp, amp_dtype, nlab, teacher=None, ada=None, log=print):
    rng = random.Random(cfg["seed"])
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    bbp = [p for n, p in named if n.startswith("bb.")]
    hp = [p for n, p in named if not n.startswith("bb.")]
    groups = ([dict(params=bbp, lr=cfg["lr"], base_lr=cfg["lr"], weight_decay=cfg["wd"])] if bbp else []) + \
             [dict(params=hp, lr=cfg["head_lr"], base_lr=cfg["head_lr"], weight_decay=cfg["wd"])]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)
    scaler = make_scaler(device, amp_dtype)
    dpe = min(len(train_docs), cfg["docs_per_epoch"])
    spe = math.ceil(dpe / (cfg["bs"] * cfg["accum"]))
    total = spe * cfg["epochs"]
    warm = max(1, int(cfg["warmup"] * total))
    eval_every = max(1, spe // max(1, cfg.get("evals_per_epoch", 1)))
    pw = torch.tensor(cfg["pos_weight"], dtype=torch.float32, device=device)
    keep = {n for n, _ in named} | {n for n, _ in model.named_buffers() if "rank_mask" in n}
    tot, tr = count_params(model)
    log(f"{len(train_docs)} train docs ({dpe}/epoch) | {spe} steps/epoch x {cfg['epochs']} = {total} steps | "
        f"params {tot / 1e6:.1f}M, trainable {tr / 1e6:.3f}M ({100 * tr / tot:.2f}%)")
    hist, best, best_state, step, t0, ntok = [], -1.0, None, 0, time.time(), 0
    stop = False
    for ep in range(cfg["epochs"]):
        ids = rng.sample(range(len(train_docs)), dpe)
        model.train()
        run, nrun = 0.0, 0
        for b0 in range(0, len(ids), cfg["bs"]):
            batch = [train_docs[i] for i in ids[b0:b0 + cfg["bs"]]]
            k = rng.randint(cfg["kmin"], cfg["kmax"])
            x, v, cm, pos = make_batch(batch, k, cfg["chunk"], cfg["select"])
            y = torch.from_numpy(to_matrix([d.labels for d in batch], nlab).astype(np.float32)).to(device)
            with amp():
                lg = model(x.to(device), v.to(device), cm.to(device), pos.to(device))
            lg = lg.float()
            loss = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
            if teacher is not None:
                T = cfg["kd_T"]
                tp = torch.tensor(np.stack([teacher[d.id] for d in batch]), dtype=torch.float32, device=device).clamp(1e-4, 1 - 1e-4)
                soft = torch.sigmoid(torch.logit(tp) / T)
                kl = F.binary_cross_entropy_with_logits(lg / T, soft) * T * T
                loss = cfg["kd_alpha"] * loss + (1 - cfg["kd_alpha"]) * kl
            if ada is not None:
                loss = loss + ada.reg_loss()
            scaler.scale(loss / cfg["accum"]).backward()
            run += loss.item(); nrun += 1; ntok += int(v.sum())
            if (b0 // cfg["bs"] + 1) % cfg["accum"] == 0 or b0 + cfg["bs"] >= len(ids):
                # cosine decay over whichever runs out first: the step schedule or the time budget
                prog = max((step - warm) / max(1, total - warm), (time.time() - t0) / 60 / cfg["max_min"])
                mult = (step + 1) / warm if step < warm else max(0.02, 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))
                for g in opt.param_groups:
                    g["lr"] = g["base_lr"] * mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for _, p in named], cfg["grad_clip"])
                if ada is not None:
                    ada.update(step)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                step += 1
                el = (time.time() - t0) / 60
                if step % 25 == 0:
                    log(f"  ep {ep + 1} step {step}/{total} loss {run / max(1, nrun):.4f} | {ntok / max(1e-6, el * 60):,.0f} tok/s | {el:.1f} min")
                stop = el > cfg["max_min"]
                if step % eval_every == 0 or stop or step == total:
                    rep, _ = evaluate(model, dev_docs, cfg, device, amp, nlab)
                    rec = dict(epoch=ep + 1, step=step, train_loss=run / max(1, nrun), minutes=el, **rep)
                    if rep["f1_global"] > best:
                        best = rep["f1_global"]
                        best_state = {kk: vv.detach().cpu().clone() for kk, vv in model.state_dict().items() if kk in keep}
                        rec["best"] = True
                    hist.append(rec)
                    log("== " + " | ".join(f"{kk} {vv:.4f}" if isinstance(vv, float) else f"{kk} {vv}" for kk, vv in rec.items()))
                    model.train()
                if stop:
                    log(f"time budget ({cfg['max_min']} min) reached - stopping")
                    break
        if stop:
            break
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        log(f"restored best checkpoint (dev-subset macro-F1 at tuned global threshold {best:.4f})")
    return hist, best


# ====================================================================== outputs
def write_prediction(ids, P, json_path, zip_path):
    """One object per unique id (first occurrence), pred = sorted unique statute ids."""
    out, seen = [], set()
    for i, row in zip(ids, P):
        if i in seen:
            continue
        seen.add(i)
        out.append({"id": i, "pred": [int(c) for c in np.where(row)[0]]})
    json.dump(out, open(json_path, "w"))
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")
    return out


def dump_probs(path, run, docs, p):
    json.dump({"run": run, "docs": {d.id: [[round(float(q), 5) for q in row], list(map(int, d.labels))] for d, row in zip(docs, p)}},
              open(path, "w"))


Writing ls_lib.py


In [4]:
%%writefile ls_peft.py
"""Adaptation methods for Phase 5, written from scratch (no `peft` library). apply(method, bb, cfg) -> info dict.
  full    : every backbone weight trainable
  partial : only the top `n_top` blocks + final norm trainable (blocks below run without autograd)
  bitfit  : bias-only tuning (Ben Zaken et al., 2022): the Phase-1 linears have no bias, so zero-initialised
            biases are added to every linear and trained together with the RMSNorm gains; all weights frozen
  lorma   : (I + (alpha/r) B A) W0 x  - multiplicative low-rank (Bihany, Patel, Modi, 2025)  [compulsory]
  adalora : P diag(lambda) Q with importance-based rank pruning + orthogonality regulariser (Zhang et al., 2023)
  adapter : Houlsby bottleneck adapters after every attention and FFN sub-layer (Houlsby et al., 2019)
  kd      : student backbone (every other layer of the teacher architecture), fully fine-tuned on gold + teacher probs
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from ls_lib import replace_linears, blocks_of, final_norm_of

DEFAULT_TARGETS = ("wq", "wk", "wv", "wo", "wg", "w_dkv", "w_ukv", "w_gate", "w_up", "w_down")


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))                 # B = 0 -> identity multiplier at init
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.register_buffer("rank_mask", torch.ones(r))

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.Q) * (self.lam * self.rank_mask), self.P) * self.scale

    def orth_loss(self):
        I = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.t() - I) ** 2).sum()


class AdaLoRAController:
    """Cubic budget schedule; importance = EMA(|w*grad|) x EMA(uncertainty), summed over the triplet (P col, lambda, Q row)."""

    def __init__(self, model, init_r, target_r, total_steps, t_init_frac=0.1, t_final_frac=0.2, delta_t=5,
                 beta1=0.85, beta2=0.85, orth_reg=0.1):
        self.layers = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.total, self.dt = total_steps, delta_t
        self.ti, self.tf = int(t_init_frac * total_steps), int(t_final_frac * total_steps)
        self.b0, self.bT = init_r * len(self.layers), target_r * len(self.layers)
        self.b1, self.b2, self.orth = beta1, beta2, orth_reg
        self.ipt = [None] * len(self.layers)
        self.unc = [None] * len(self.layers)

    def reg_loss(self):
        return self.orth * sum(m.orth_loss() for m in self.layers) / len(self.layers)

    def budget(self, step):
        end = self.total - self.tf
        if step < self.ti:
            return self.b0
        if step >= end:
            return self.bT
        return int(self.bT + (self.b0 - self.bT) * (1 - (step - self.ti) / max(1, end - self.ti)) ** 3)

    @torch.no_grad()
    def update(self, step):
        scores = []
        for i, m in enumerate(self.layers):
            g = lambda p: p.grad if p.grad is not None else torch.zeros_like(p)
            s = torch.stack([(m.lam * g(m.lam)).abs(), (m.P * g(m.P)).abs().mean(0), (m.Q * g(m.Q)).abs().mean(1)]).float()
            s = torch.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
            if self.ipt[i] is None:
                self.ipt[i], self.unc[i] = torch.zeros_like(s), torch.zeros_like(s)
            self.ipt[i] = self.b1 * self.ipt[i] + (1 - self.b1) * s
            self.unc[i] = self.b2 * self.unc[i] + (1 - self.b2) * (s - self.ipt[i]).abs()
            scores.append((self.ipt[i] * self.unc[i]).sum(0))
        end = self.total - self.tf
        if step < self.ti or step > end or ((step - self.ti) % self.dt != 0 and step != end):
            return
        b = self.budget(step)
        allsc = torch.cat(scores)
        if b >= allsc.numel():
            return
        thr = torch.topk(allsc, b).values[-1]
        for m, sc in zip(self.layers, scores):
            m.rank_mask.copy_((sc >= thr).float())

    def ranks(self):
        return [int(m.rank_mask.sum().item()) for m in self.layers]


class Adapter(nn.Module):
    """Wraps a sub-layer (attention or FFN): out = f(x); out + up(gelu(down(out))). Up-projection starts at 0."""

    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.down, self.up = nn.Linear(d, m), nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        h = self.inner(*a, **k)
        return h + self.up(F.gelu(self.down(h)))


def apply(method, bb, cfg):
    """The caller has already set requires_grad=False on every backbone parameter."""
    info = {"method": method}
    nL = len(blocks_of(bb))
    if method in ("full", "kd"):
        for p in bb.parameters():
            p.requires_grad = True
        n = sum(1 for _ in bb.parameters())
    elif method == "partial":
        top = cfg["n_top"]
        for blk in list(blocks_of(bb))[nL - top:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in final_norm_of(bb).parameters():
            p.requires_grad = True
        info["train_from"] = nL - top
        n = top
    elif method == "bitfit":
        n = 0
        for blk in blocks_of(bb):
            for mod in blk.modules():
                if isinstance(mod, nn.Linear):
                    if mod.bias is None:
                        mod.bias = nn.Parameter(torch.zeros(mod.out_features))
                    mod.bias.requires_grad = True
                    n += 1
                elif type(mod).__name__ in ("RMSNorm", "LayerNorm"):
                    for p in mod.parameters():
                        p.requires_grad = True
        for p in final_norm_of(bb).parameters():
            p.requires_grad = True
    elif method == "lorma":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: LoRMALinear(l, cfg["r"], cfg["alpha"]))
    elif method == "adalora":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: AdaLoRALinear(l, cfg["init_r"], cfg["alpha"]))
    elif method == "adapter":
        d, m = bb.cfg.d_model, cfg["bottleneck"]
        n = 0
        for blk in blocks_of(bb):
            blk.attn = Adapter(blk.attn, d, m)
            blk.ffn = Adapter(blk.ffn, d, m)
            n += 2
        for blk in blocks_of(bb):                                  # Houlsby: layer norms are trained as well
            for mod in blk.modules():
                if type(mod).__name__ in ("RMSNorm", "LayerNorm"):
                    for p in mod.parameters():
                        p.requires_grad = True
    else:
        raise ValueError(f"unknown method {method}")
    assert n > 0, f"{method}: nothing to adapt"
    info["n_modules"] = n
    print(f"[{method}] applied ({n} modules)", {k: v for k, v in cfg.items() if k != "targets"})
    return info


Writing ls_peft.py


In [5]:
%%writefile ls_run.py
"""Phase 5 LSI - one experiment. Usage: python ls_run.py <method> <arch> [tag]
method: full | partial | bitfit | lorma | adalora | adapter | kd      arch: gqa | mla | retnet | encdec
Env: LS_SMOKE=1 (tiny 1-epoch check), LS_OVERRIDES='{"SELECT": "tail"}' (override any upper-case setting below)."""
import os, sys, json, math, time, random, glob
from pathlib import Path
import numpy as np
import torch

METHOD, ARCH = sys.argv[1], sys.argv[2]
TAG = sys.argv[3] if len(sys.argv) > 3 else ""
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SEED = 42

# ---- long-document input
CHUNK = 512
SELECT = "headtail"        # headtail: first chunks + the document's final chunk | head | tail (ablation)
KMIN, KMAX = 2, 4          # training: random number of chunks per batch
K_INFER = 4                # inference / dev: up to 4 x 512 = 2048 tokens
KEEP_TOK = 3584            # head tokens kept per document (7 chunks)

# ---- optimisation
EPOCHS = 4
DOCS_PER_EPOCH = 12000
BS, ACCUM, EVAL_BS = 4, 4, 16
HEAD_LR, WD, WARMUP, GRAD_CLIP = 1e-3, 0.01, 0.05, 1.0
POS_W_POW, POS_W_MAX = 0.5, 15.0   # BCE positive weight per statute = clip((neg/pos)^POW, 1, MAX): rare statutes weigh more
EVALS_PER_EPOCH = 2
MAX_MIN = 75               # training time budget per run (minutes); the best dev checkpoint is kept
HCFG = dict(hid=384, dropout=0.15, ckpt=True)
KD_DOCS = 8000             # fixed subset of training docs on which every run writes probabilities (KD teacher signal)
DEV_SEL_N = 1500           # fixed dev subset used for checkpoint selection

MCFG_ALL = {
    "full":    dict(lr=2e-5),
    "partial": dict(lr=5e-5, n_top=6),
    "bitfit":  dict(lr=1e-3),
    "lorma":   dict(lr=5e-4, r=16, alpha=16),
    "adalora": dict(lr=1e-3, init_r=12, target_r=6, alpha=12, orth_reg=0.1, delta_t=5),
    "adapter": dict(lr=5e-4, bottleneck=64),
    "kd":      dict(lr=3e-5, keep=None, alpha=0.3, T=2.0),     # student = every other layer, fully fine-tuned
}
for _k, _v in json.loads(os.environ.get("LS_OVERRIDES", "{}")).items():
    globals()[_k] = _v
SMOKE = os.environ.get("LS_SMOKE") == "1"
if SMOKE:
    EPOCHS, DOCS_PER_EPOCH, MAX_MIN, KD_DOCS, DEV_SEL_N = 1, 48, 5, 40, 64
MCFG = dict(MCFG_ALL[METHOD])

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK = ROOT / f"ls_{METHOD}_{ARCH}{TAG}{'_smoke' if SMOKE else ''}"
WORK.mkdir(parents=True, exist_ok=True)
print("device:", device, "| out:", WORK, "| cfg:", MCFG, "| select:", SELECT, flush=True)

import ls_lib, ls_peft
from ls_metrics import to_matrix, choose, decide, freq_buckets, per_label_f1
from huggingface_hub import hf_hub_download, HfApi
from tokenizers import Tokenizer
amp, AMP_DTYPE = ls_lib.get_amp(device)

# ------------------------------------------------------------------ data
files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
p5 = [f for f in files if "Phase_5" in f or "LSI" in f]
print("Phase-5 files:", p5)
def fetch(name):
    m = [f for f in p5 if f.endswith("/" + name) or f == name]
    assert m, f"{name} not found in {p5}"
    return hf_hub_download(HF_DATASET_REPO, m[0], repo_type="dataset", token=HF_TOKEN)

recs = {}
for part in ("train", "dev", "test"):
    recs[part], raw = ls_lib.parse_lsi(fetch(f"{part}.json"))
    r0 = recs[part][0]
    print(f"{part}: {len(recs[part])} docs | raw {type(raw).__name__} | keys {list(raw[0].keys()) if isinstance(raw, list) and isinstance(raw[0], dict) else '-'} "
          f"| example id={r0[0]!r} labels={r0[2]} text={r0[1][:120]!r}", flush=True)
seen, tr = set(), []
for r in recs["train"]:
    if r[0] not in seen and r[2]:
        seen.add(r[0]); tr.append(r)
print(f"train after id de-duplication / dropping unlabelled: {len(tr)} (removed {len(recs['train']) - len(tr)})")
recs["train"] = tr
assert all(r[2] for r in recs["dev"]), "every dev document must have labels"
NLAB = 1 + max(max(r[2]) for p in ("train", "dev") for r in recs[p])
print(f"max label id: train {max(max(r[2]) for r in recs['train'])} | dev {max(max(r[2]) for r in recs['dev'])} -> NLAB {NLAB}")
if SMOKE:
    recs["train"], recs["dev"] = recs["train"][:200], recs["dev"][:64]

export_dir = ls_lib.find_p1_export(ARCH)
assert export_dir is not None, f"Phase-1 export for {ARCH} not found under /kaggle/input"
tokenizer = Tokenizer.from_file(str(export_dir / "tokenizer.json"))
t0 = time.time()
docs = {p: ls_lib.tokenize(recs[p], tokenizer, KEEP_TOK, CHUNK, SELECT) for p in recs}
print(f"tokenised in {time.time() - t0:.0f}s", flush=True)
train_docs, dev_docs, test_docs = docs["train"], docs["dev"], docs["test"]
Ytr = to_matrix([d.labels for d in train_docs], NLAB)
Ydev = to_matrix([d.labels for d in dev_docs], NLAB)
cnt = Ytr.sum(0)
for p in ("train", "dev", "test"):
    w = np.array([d.nwords for d in docs[p]]); tk = np.array([d.ntok_est for d in docs[p]])
    print(f"{p}: words/doc mean {w.mean():.0f} median {np.median(w):.0f} p90 {np.percentile(w, 90):.0f} max {w.max()} | "
          f"head tokens (capped {KEEP_TOK * 7} chars) median {np.median(tk):.0f} | "
          f"docs > {K_INFER * CHUNK} tok ~{(tk > K_INFER * CHUNK).mean():.2f}" +
          (f" | labels/doc mean {np.mean([len(d.labels) for d in docs[p]]):.2f} max {max(len(d.labels) for d in docs[p])}" if p != "test" else ""))
print(f"labels: NLAB {NLAB} | used in train {(cnt > 0).sum()} | in dev {(Ydev.sum(0) > 0).sum()} | dev labels never in train "
      f"{int(((Ydev.sum(0) > 0) & (cnt == 0)).sum())} | train count min {cnt[cnt > 0].min()} median {np.median(cnt[cnt > 0]):.0f} max {cnt.max()} | "
      f"labels with <20 train docs {int(((cnt > 0) & (cnt < 20)).sum())}, <100 {int(((cnt > 0) & (cnt < 100)).sum())}")
tkeys = {d.key for d in train_docs}
print(f"overlap with train (same text head): dev {sum(d.key in tkeys for d in dev_docs)} | test {sum(d.key in tkeys for d in test_docs)} | "
      f"test unique ids {len({d.id for d in test_docs})}/{len(test_docs)}")
n = len(train_docs)
POS_W = np.clip(((n - cnt) / np.maximum(cnt, 1)) ** POS_W_POW, 1.0, POS_W_MAX)
POS_W[cnt == 0] = 1.0
PRIOR = np.maximum(cnt, 0.5) / n
kd_rng = random.Random(1234)
KD_IDS = set(kd_rng.sample([d.id for d in train_docs], min(KD_DOCS, len(train_docs))))

# ------------------------------------------------------------------ model
arch_mod, bcfg, bb = ls_lib.load_p1(export_dir)
P1_PARAMS = sum(p.numel() for p in bb.parameters())
print(f"Phase-1 {ARCH}: {P1_PARAMS / 1e6:.1f}M params, kind {bb.kind}, positions {ls_lib.max_positions(bb)}")
TEACHER, TEACHER_INFO = None, None
if METHOD == "kd":
    tfiles = sorted(set(glob.glob(str(ROOT / "ls_*/probs_kd.json")) + glob.glob("/kaggle/input/**/probs_kd.json", recursive=True)))
    tfiles = [f for f in tfiles if ("_smoke" in f) == SMOKE and "_tail" not in f and Path(f).parent.resolve() != WORK.resolve()]
    assert tfiles, "KD needs probs_kd.json from earlier runs"
    acc, TEACHER_INFO = {}, []
    for f in tfiles:
        j = json.load(open(f))
        res = Path(f).with_name("results.json")
        TEACHER_INFO.append(dict(run=j["run"], **({k: json.load(open(res)).get(k) for k in ("dev_macro", "params_total")} if res.exists() else {})))
        for k, (p, _) in j["docs"].items():
            acc.setdefault(k, []).append(p)
    TEACHER = {k: np.mean(v, 0).astype(np.float32) for k, v in acc.items() if len(v) == len(tfiles)}
    print("teacher = mean of", [t["run"] for t in TEACHER_INFO], "|", len(TEACHER), "docs")
    train_docs = [d for d in train_docs if d.id in TEACHER]
    nL = len(ls_lib.blocks_of(bb))
    keep = MCFG["keep"] or list(range(0, nL, 2))
    bb, bcfg = ls_lib.make_student_bb(arch_mod, bcfg, bb, keep)
    print(f"student keeps layers {keep} of {nL}: {sum(p.numel() for p in bb.parameters()) / 1e6:.1f}M params")

ls_lib.strip_backbone(bb)
ls_lib.patch_encoder_padding_mask(bb)          # must happen before adapters wrap the attention modules
for p in bb.parameters():
    p.requires_grad = False
INFO = ls_peft.apply(METHOD, bb, MCFG)
assert CHUNK <= ls_lib.max_positions(bb)
HCFG["train_from"] = INFO.get("train_from", 0)
model = ls_lib.LSIModel(bb, HCFG, NLAB, PRIOR).to(device)
tot, trn = ls_lib.count_params(model)
print(f"model {tot / 1e6:.1f}M params | trainable {trn / 1e6:.3f}M | train_from {HCFG['train_from']}", flush=True)

TCFG = dict(lr=MCFG["lr"], head_lr=HEAD_LR, wd=WD, warmup=WARMUP, epochs=EPOCHS, docs_per_epoch=DOCS_PER_EPOCH, bs=BS,
            accum=ACCUM, kmin=KMIN, kmax=KMAX, k_infer=K_INFER, chunk=CHUNK, eval_bs=EVAL_BS, grad_clip=GRAD_CLIP,
            pos_weight=POS_W.tolist(), max_min=MAX_MIN, seed=SEED, evals_per_epoch=EVALS_PER_EPOCH, select=SELECT,
            kd_alpha=MCFG.get("alpha", 1.0), kd_T=MCFG.get("T", 1.0))
ADA = None
if METHOD == "adalora":
    spe = math.ceil(min(len(train_docs), DOCS_PER_EPOCH) / (BS * ACCUM))
    ADA = ls_peft.AdaLoRAController(model, MCFG["init_r"], MCFG["target_r"], spe * EPOCHS, delta_t=MCFG["delta_t"], orth_reg=MCFG["orth_reg"])
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
DEV_SEL = random.Random(7).sample(dev_docs, min(DEV_SEL_N, len(dev_docs)))
hist, best = ls_lib.train(model, train_docs, DEV_SEL, TCFG, device, amp, AMP_DTYPE, NLAB, teacher=TEACHER, ada=ADA)
TRAIN_MIN = (time.time() - t0) / 60
PEAK = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
if ADA is not None:
    print("AdaLoRA final ranks:", ADA.ranks())

# ------------------------------------------------------------------ dev, test, outputs
p_dev = ls_lib.predict(model, dev_docs, K_INFER, CHUNK, EVAL_BS, device, amp, SELECT)
ch = choose(Ydev, p_dev)
P_dev = ch["P"]
print(f"DEV macro-F1 {ch['macro']:.4f} (rule {ch['rule']}) | honest {{{', '.join(f'{k}: {v:.4f}' for k, v in ch['honest'].items())}}} | "
      f"micro {ch['micro']:.4f} | subset acc {ch['subset']:.4f} | macro P/R {ch['macro_p_r'][0]:.3f}/{ch['macro_p_r'][1]:.3f} | "
      f"labels/doc pred {P_dev.sum(1).mean():.2f} gold {Ydev.sum(1).mean():.2f}")
fb = freq_buckets(Ytr, Ydev, P_dev)
print("DEV macro-F1 by train frequency:", {k: (v[0], None if v[1] is None else round(v[1], 4)) for k, v in fb.items()})
t1 = time.time()
p_test = ls_lib.predict(model, test_docs, K_INFER, CHUNK, EVAL_BS, device, amp, SELECT)
SPS = len(test_docs) / max(1e-6, time.time() - t1)
P_test = decide(p_test, ch["thr"])
P_test[:, cnt == 0] = False                     # never predict a statute absent from training
ls_lib.write_prediction([d.id for d in test_docs], P_test, WORK / "prediction.json", WORK / "submission.zip")
print(f"test: {len(test_docs)} docs | labels/doc {P_test.sum(1).mean():.2f} | {SPS:.1f} docs/s")
_seen = set()
json.dump([{"id": d.id, "gold": list(map(int, d.labels))} for d in dev_docs if not (d.id in _seen or _seen.add(d.id))], open(WORK / "dev_truth.json", "w"))
ls_lib.write_prediction([d.id for d in dev_docs], P_dev, WORK / "dev_prediction.json", WORK / "dev_submission.zip")
ls_lib.dump_probs(WORK / "probs_dev.json", WORK.name, dev_docs, p_dev)
ls_lib.dump_probs(WORK / "probs_test.json", WORK.name, test_docs, p_test)
if METHOD != "kd" and SELECT != "tail":
    kd_docs = [d for d in docs["train"] if d.id in KD_IDS]
    ls_lib.dump_probs(WORK / "probs_kd.json", WORK.name, kd_docs, ls_lib.predict(model, kd_docs, K_INFER, CHUNK, EVAL_BS, device, amp, SELECT))
plf = per_label_f1(Ydev, P_dev)
res = dict(run=WORK.name, method=METHOD, arch=ARCH, select=SELECT, mcfg=MCFG, hcfg=HCFG, info=INFO, params_total=tot, params_trainable=trn,
           phase1_params=P1_PARAMS, train_min=TRAIN_MIN, peak_gpu_gb=PEAK, test_docs_per_s=SPS, chunk=CHUNK, k_infer=K_INFER, nlab=NLAB,
           dev_macro=ch["macro"], dev_micro=ch["micro"], dev_subset=ch["subset"], dev_macro_p_r=ch["macro_p_r"], rule=ch["rule"],
           honest=ch["honest"], thr=np.asarray(ch["thr"]).tolist(), freq_buckets=fb, per_label_f1=plf.tolist(),
           train_label_count=cnt.tolist(), dev_label_count=Ydev.sum(0).tolist(), history=hist, teacher=TEACHER_INFO, n_train=len(train_docs))
json.dump(res, open(WORK / "results.json", "w"), indent=1, default=float)
print(f"\nSUMMARY {WORK.name}: dev macro-F1 {ch['macro']:.4f} ({ch['rule']}, honest {max(ch['honest'].values()):.4f}) | micro {ch['micro']:.4f} | "
      f"subset {ch['subset']:.4f} | params {tot / 1e6:.1f}M trainable {trn / 1e6:.2f}M | train {TRAIN_MIN:.1f} min | peak {PEAK:.1f} GB | {SPS:.1f} docs/s")
if METHOD == "kd":
    tf = [t.get("dev_macro") for t in TEACHER_INFO if t.get("dev_macro")]
    tp = [t.get("params_total") for t in TEACHER_INFO if t.get("params_total")]
    if tf:
        print(f"KD: student reaches {100 * ch['macro'] / max(tf):.1f}% of the best teacher's dev macro-F1 ({max(tf):.4f})"
              + (f" with {100 * tot / max(tp):.0f}% of its parameters" if tp else ""))
print("RUN_DONE", WORK.name, flush=True)


Writing ls_run.py


In [6]:
%%writefile ls_ens.py
"""Phase 5 - ensemble of finished runs: mean statute probabilities, greedy forward selection on dev
(criterion = honest macro-F1, i.e. thresholds fitted on one half of dev and scored on the other), then the
threshold rule refitted on all of dev. Writes submissions/ensemble_greedy.zip and submissions/ensemble_all.zip."""
import json, glob, sys, os
from pathlib import Path
import numpy as np
sys.path.insert(0, ".")
from ls_metrics import choose, decide, to_matrix, freq_buckets
from ls_lib import write_prediction

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUB = ROOT / "submissions"; SUB.mkdir(exist_ok=True)
SMOKE = "--smoke" in sys.argv          # pipeline check on the smoke runs (writes smoke_* zips)
dirs = sorted({str(Path(p).parent) for p in glob.glob(str(ROOT / "ls_*/probs_test.json"))
               if ("_smoke" in p) == SMOKE and Path(p).with_name("probs_dev.json").exists()})
runs = {}
for d in dirs:
    jd, jt = json.load(open(Path(d) / "probs_dev.json")), json.load(open(Path(d) / "probs_test.json"))
    runs.setdefault(jd["run"], (jd["docs"], jt["docs"]))
names = sorted(runs)
assert names, "no runs found"
dev_ids = sorted(set.intersection(*[set(runs[n][0]) for n in names]))
test_ids = sorted(set.intersection(*[set(runs[n][1]) for n in names]))
nlab = len(next(iter(runs[names[0]][0].values()))[0])
Y = to_matrix([runs[names[0]][0][i][1] for i in dev_ids], nlab)
PD = {n: np.array([runs[n][0][i][0] for i in dev_ids], dtype=np.float32) for n in names}
PT = {n: np.array([runs[n][1][i][0] for i in test_ids], dtype=np.float32) for n in names}
tr_cnt = None
for d in dirs:
    r = Path(d) / "results.json"
    if r.exists():
        tr_cnt = np.array(json.load(open(r))["train_label_count"]); break
print(f"dev docs {len(dev_ids)} | test docs {len(test_ids)} | labels {nlab}")
score = {}
for n in names:
    c = choose(Y, PD[n]); score[n] = max(c["honest"].values())
    print(f"{n:>26}  macro {c['macro']:.4f} ({c['rule']})  honest {score[n]:.4f}  micro {c['micro']:.4f}")

members, best = [], -1.0
while True:
    cand = []
    for n in names:
        c = choose(Y, np.mean([PD[m] for m in members + [n]], 0))
        cand.append((max(c["honest"].values()), n))
    f, n = max(cand)
    if f <= best + 1e-5 or len(members) >= 8:
        break
    members.append(n); best = f
    print(f"+ {n:>26} -> honest dev macro-F1 {f:.4f}")


def build(mem, name):
    pd_, pt_ = np.mean([PD[m] for m in mem], 0), np.mean([PT[m] for m in mem], 0)
    c = choose(Y, pd_)
    P = decide(pt_, c["thr"])
    if tr_cnt is not None:
        P[:, tr_cnt[:nlab] == 0] = False
    name = ("smoke_" if SMOKE else "") + name
    write_prediction(test_ids, P, ROOT / f"prediction_{name}.json", SUB / f"{name}.zip")
    print(f"{name.upper()} {mem}: dev macro {c['macro']:.4f} ({c['rule']}) | honest {c['honest']} | micro {c['micro']:.4f} | "
          f"subset {c['subset']:.4f} | macro P/R {c['macro_p_r'][0]:.3f}/{c['macro_p_r'][1]:.3f} | test labels/doc {P.sum(1).mean():.2f}")
    if tr_cnt is not None:
        print("   by train frequency:", {k: (v[0], None if v[1] is None else round(v[1], 4)) for k, v in freq_buckets(tr_cnt, Y, c["P"]).items()})
    return dict(members=mem, dev_macro=c["macro"], honest=c["honest"], rule=c["rule"], micro=c["micro"], subset=c["subset"])


out = {"greedy": build(members, "ensemble_greedy"), "all": build([n for n in names if "_tail" not in n], "ensemble_all")}
json.dump(out, open(ROOT / "ensemble_results.json", "w"), indent=1)
print("ENS_DONE")


Writing ls_ens.py


In [7]:
# ===== Phase 5 driver: every experiment is its own process, one queue per T4; then KD (needs the teachers) + a
# section ablation; then the ensemble. Set SMOKE = True for a ~15-minute pipeline check first.
import os, sys, subprocess, time, glob, shutil, json, py_compile
from pathlib import Path
import torch
for f in ("ls_metrics.py", "ls_lib.py", "ls_peft.py", "ls_run.py", "ls_ens.py"):
    py_compile.compile(f, doraise=True)
SMOKE = False        # True = tiny 1-epoch pipeline check. Set False for the real runs (~6 h on 2 x T4).
os.environ["HF_TOKEN"] = HF_TOKEN or ""
NGPU = max(1, torch.cuda.device_count())
# (method, arch, tag, overrides)
QUEUES = {0: [("full", "gqa", "", {}), ("lorma", "gqa", "", {}), ("bitfit", "mla", "", {})],
          1: [("partial", "retnet", "", {}), ("adapter", "encdec", "", {}), ("adalora", "retnet", "", {})]}
FINAL = {0: [("kd", "gqa", "", {})],
         1: [("lorma", "gqa", "_tail", {"SELECT": "tail"})]}          # ablation: last 4 chunks instead of head + end
if NGPU == 1:
    QUEUES = {0: QUEUES[0] + QUEUES[1]}
    FINAL = {0: FINAL[0] + FINAL[1]}
LOG = Path("/kaggle/working/logs"); LOG.mkdir(parents=True, exist_ok=True)
T0 = time.time()

def tail(path, n=1):
    try:
        return open(path).read().strip().splitlines()[-n:]
    except Exception:
        return []

def run_queues(queues):
    pend = {g: list(q) for g, q in queues.items()}
    cur, status, last = {g: None for g in queues}, {}, 0
    while True:
        for g in queues:
            if cur[g] is not None and cur[g][0].poll() is not None:
                p, f, name = cur[g]; f.close(); status[name] = p.returncode
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} finished {name} rc={p.returncode}\n    " + "\n    ".join(tail(LOG / f"{name}.log", 4)), flush=True)
                cur[g] = None
            if cur[g] is None and pend[g]:
                m, a, tag, ov = pend[g].pop(0); name = f"{m}_{a}{tag}"
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), LS_SMOKE="1" if SMOKE else "0", PYTHONUNBUFFERED="1",
                           LS_OVERRIDES=json.dumps(ov))
                f = open(LOG / f"{name}.log", "w")
                cur[g] = (subprocess.Popen([sys.executable, "ls_run.py", m, a, tag], stdout=f, stderr=subprocess.STDOUT, env=env), f, name)
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} started {name}", flush=True)
        if all(c is None for c in cur.values()):
            return status
        if time.time() - last > 300:
            last = time.time()
            for g, c in cur.items():
                if c is not None:
                    print(f"[{(time.time() - T0) / 60:6.1f} min]   GPU{g} {c[2]}: {tail(LOG / (c[2] + '.log'))}", flush=True)
        time.sleep(15)

print("GPUs:", NGPU, "| queues:", QUEUES, "| smoke:", SMOKE, flush=True)
status = run_queues(QUEUES)
status.update(run_queues(FINAL))
print("\nreturn codes:", status)
sub = Path("/kaggle/working/submissions"); sub.mkdir(exist_ok=True)
for z in glob.glob("/kaggle/working/ls_*/submission.zip"):
    p = Path(z); shutil.copy(z, sub / f"{p.parent.name}.zip")
r = subprocess.run([sys.executable, "ls_ens.py"] + (["--smoke"] if SMOKE else []), capture_output=True, text=True)
print(r.stdout[-20000:], r.stderr[-5000:])
print(sorted(os.listdir(sub)))
print("\n=== SUMMARIES ===")
for lg in sorted(LOG.glob("*.log")):
    for line in open(lg).read().splitlines():
        if line.startswith(("SUMMARY", "KD:", "DEV ", "overlap", "train:", "dev:", "test:", "labels:", "max label", "AdaLoRA final")) \
                or "Traceback" in line or "Error" in line:
            print(lg.stem, "|", line)
print("ALL_DONE")


GPUs: 2 | queues: {0: [('full', 'gqa', '', {}), ('lorma', 'gqa', '', {}), ('bitfit', 'mla', '', {})], 1: [('partial', 'retnet', '', {}), ('adapter', 'encdec', '', {}), ('adalora', 'retnet', '', {})]} | smoke: False
[   0.0 min] GPU0 started full_gqa
[   0.0 min] GPU1 started partial_retnet
[   0.0 min]   GPU0 full_gqa: []
[   0.0 min]   GPU1 partial_retnet: []
[   5.0 min]   GPU0 full_gqa: ['  ep 1 step 50/3000 loss 0.5524 | 8,553 tok/s | 1.2 min']
[   5.0 min]   GPU1 partial_retnet: ['  ep 1 step 50/3000 loss 0.5507 | 9,875 tok/s | 1.1 min']
[  10.0 min]   GPU0 full_gqa: ['  ep 1 step 225/3000 loss 0.4290 | 8,001 tok/s | 5.7 min']
[  10.0 min]   GPU1 partial_retnet: ['  ep 1 step 275/3000 loss 0.4199 | 9,510 tok/s | 5.8 min']
[  15.0 min]   GPU0 full_gqa: ['  ep 1 step 400/3000 loss 0.3968 | 7,439 tok/s | 10.8 min']
[  15.0 min]   GPU1 partial_retnet: ['  ep 1 step 475/3000 loss 0.3936 | 8,645 tok/s | 11.1 min']
[  20.0 min]   GPU0 full_gqa: ['  ep 1 step 600/3000 loss 0.3776 | 7,588 